In [ ]:
# Full name test
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

In [ ]:
#@title install required packages, download test files, initialize Otter

# install packages
%pip install otter-grader==5.5.0

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import requests, os

    os.makedirs('tests', exist_ok=True)
    os.makedirs('data', exist_ok=True)
    
    api_tests = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A1_introduction/tests"
    api_data = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A1_introduction/data"

    # download tests
    for f in requests.get(api_tests).json():
        open(f"tests/{f['name']}", 'w').write(requests.get(f['download_url']).text)

    # download data
    for f in requests.get(api_data).json():
        open(f"data/{f['name']}", 'wb').write(requests.get(f['download_url']).content)

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('3_tutorial_data_science.ipynb')

***

# Rudiments of Data Science: Analysing the Curation of the MoMA

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 23.04.2026
+ **Authors:** [Dr. Benedikt Zönnchen](https://bzoennchen.github.io/Pages/) & [Dr. Téo Sanchez](https://teo-sanchez.github.io/)

⛳ **By the end of this notebook you will be able to:**
- Work with `numpy` arrays and understand why they are more efficient than plain Python lists for numerical data
- Create and manipulate `pandas` `Series` and `DataFrame` objects to represent tabular data
- Compute basic statistics on a dataset (mean, median, min, max) and extract metadata
- Filter and slice data using boolean masking
- Visualize data distributions and trends using `seaborn` and `matplotlib`
- Load real-world data from CSV files, clean it, merge two datasets, and draw data-driven conclusions

**⚠️ Before you start — important notes for Google Colab:**
- **Save your work first:** Go to *File → Save a copy in Drive* to save this notebook to your Google Drive. After that, Colab will auto-save your progress. You can also download your work at any time via *File → Download → Download .ipynb*❗
- **Run cells in order, from top to bottom.** Each cell can only use variables defined in cells that have already been run❗ 
- **Got an unexpected error?** Go to *Runtime → Restart session and run all* to start fresh❗ 
- **Colab disconnects after ~30 minutes of inactivity**, which erases all your variables. If this happens, go to *Runtime → Restart session and run all*. Your saved notebook in Drive is not affected❗ 

This notebook is a brief introduction to data science. It is intended for beginners who are interested in the manipulation of data and the process of answering research questions using data.

The notebook is interested in a concrete application: **the analysis of the artistic curation of the Museum of Modern Art (MoMA) in New York**.

<image src="https://live.staticflickr.com/159/430975967_c97c2d8e35_b.jpg" style="width: 700px; display: block; margin-left: auto; margin-right: auto;"/>

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/3_tutorial_data_science.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In the previous tutorial, you learned about two native data structures in python: lists (`list`) and dictionaries (`dict`). 
Even if thoses data structure are essential in python, they lack functionalities to manipulate data in a more efficient way.


With the package `pandas`, we are going to learn how to manipulate data frames, i.e. a table with *attributes* as columns and *instances* as rows.

## 8 The `numpy` Package

[numpy](https://numpy.org/) -- also NumPy -- (Numerical Python) is a fundamental Python library for numerical computing. It provides support for large, multi-dimensional **arrays** and **matrices**, along with a collection of mathematical functions to operate on these arrays efficiently.

Unlike `Python` lists, `numpy` arrays are **homogeneous**, meaning all elements must be of the same data type (e.g., all integers or all floats). This homogeneity, combined with `numpy`'s optimized C backend, makes numpy` operations significantly faster than equivalent Python operations, especially for large datasets.

A `numpy` array is created using the `numpy.ndarray` type. For example, a one-dimensional array is called a **vector**, and a two-dimensional array is called a **matrix**. NumPy arrays form the foundation of many higher-level data analysis tools, including pandas DataFrames, which build upon NumPy's array structure.

Common operations include element-wise arithmetic, linear algebra, statistical functions, and array manipulation. NumPy is essential for scientific computing, data analysis, and machine learning workflows in Python.

By using `numpy`, we can very easily work with lists of numbers like mathematical vectors and lists of lists like matrices. In `numpy` we call these lists ``arrays``.

For example, let us define two vectors $a$ and $b$ and then compute the dot product of these vectors:

In [ ]:
import numpy as np

a = np.array([[1,2,3,4]])
b = np.array([[-1, 0, 0, 1]])

dot_product = a @ b.T
dot_product

This computes $$\begin{bmatrix} 1 & 2 & 3 & 4 \end{bmatrix} \cdot \begin{bmatrix} -1 \\ 0 \\ 0 \\ 1 \end{bmatrix} = 1 \cdot (-1) + 2 \cdot 0 + 3 \cdot 0 + 4 \cdot 1 = -1+4 = 3$$

Instead of $a \cdot b^\top$ we can also compute $a^\top \cdot b$ which results in a matrix:

In [ ]:
matrix = a.T @ b
matrix

Machine learning is all about matrix-matrix and matrix-vector multiplication! ``numpy`` is highly optimized for these operations.

## 9 The `pandas` Package

[pandas](https://pandas.pydata.org/) (Python Data Analysis Library) is a powerful library for data manipulation and analysis. It provides high-level data structures and tools designed to make working with tabular and time-series data fast and intuitive.

The core data structures in pandas are the `Series` (one-dimensional) and `DataFrame` (two-dimensional). Unlike `numpy` arrays, which must be homogeneous, pandas data structures are **heterogeneous** -- a single DataFrame can contain columns of different data types (integers, floats, strings, dates, etc.), making it ideal for real-world datasets. You can think of a `DataFrame` as an (Excel) spreadsheet or SQL table, where each column is labeled and rows can also have labels (an index).

`pandas` builds on top of `numpy`, inheriting its computational efficiency while adding a layer of abstraction that makes data analysis more accessible. Common operations include filtering, grouping, aggregating, merging, and reshaping data. Pandas also provides seamless integration with other libraries for visualization (`matplotlib`, `seaborn`) and statistical analysis (`scikit-learn`).

`pandas` is the de facto standard for data manipulation in `Python` and is essential for exploratory data analysis, data cleaning, and preparation before machine learning or statistical modeling.

### 9.1 Series (`pandas.Series`)

A serie (`pandas.Series`) is a one-dimensional array that can store any data type (integers, strings, floats, etc.). It is similar to a list in python, but with more functionalities. 

A serie can be instanciated from a Python `list`.

Let's create a serie of artworks from the MoMA collection.

In [ ]:
# We can import the package - basically meaning copying the code - by the following command
import pandas as pd

In [ ]:
artworks = pd.Series(["Spiral for Shared Dreams",
                      "Woman with Dead Child",
                      "Self-Portrait en Face",
                      "Standard Station, Ten-Cent Western Being Torn in Half",
                      "Houston Street",
                      "Shooting Painting American Embassy",
                      "Self-Portrait with Cropped Hair",
                      "Patchwork Quilt"
])

artworks

The serie is displayed as a column. The rows are displayed alongside indexes $0,1,\ldots$. It is possible to reassign indexes, even to non-integers. 

The notation `serie[i]` is used to access the row with index `i`:

In [ ]:
artworks[5]

As with Python lists, the length of a series can be accessed with the `len` function:

In [ ]:
len(artworks)

Let us create an hypothetical series of prices for the artworks stored in the `artworks` variable (in thousands of dollars \$)

In [ ]:
prices = pd.Series([500, 300, 250, 5000, 1000, 1200, 10000, 700])
prices

Because this serie is numerical, we can perform operations on it. For example, we can calculate the average price of the artworks:

In [ ]:
prices.mean()

<!-- BEGIN QUESTION -->

---

**Exercise 9.1:** Now calculate the median price using the `median` method.

---

In [ ]:
median = ...
median

In [ ]:
grader.check("q91")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 9.2:** Calculate the minimum and maximum price of the serie using the `min` resp. `max` method.

---

In [ ]:
price_min = ...
price_max = ...

print(price_min)
print(price_max)

In [ ]:
grader.check("q92")

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 9.3:** What is the range = *maximum - minimum* of prices.

---

In [ ]:
price_range = ...
price_range

In [ ]:
grader.check("q93")

<!-- END QUESTION -->

### 9.2 Data frames (`pandas.DataFrame`)

*Data frames* (type `pandas.DataFrame`) are arrays of two dimensions, with labels (*labels*) associated to rows and columns. You can imagine it as an *Excelsheet*. Different from `numpy` *arrays* (and later `torch` *tensors*) which can only consist of numbers, a data frame is not necessarily **homogeneous** as data types can change from one column to another (strings, floats, integer, etc.).


In the rest of the course, by **table**, we mean a two-dimensional array of type `pandas.DataFrame`, while by **series** we mean a one-dimensional array of type `pandas.Series`.

These concepts of `DataFrame` and `Series` can also be found in other packages or data analysis systems such as [`R`](https://www.r-project.org/).

Using `pandas` dataframes can be helpful for all kinds of analysis and purposes. For instance, suppose you want to keep track of your personal **book collection**. You could use a `DataFrame` where each row represents a book, with columns for the title, author, genre, publication year, and whether you've read it. Unlike a simple list, a `DataFrame` lets you easily filter (e.g., "show all science fiction books"), calculate statistics (e.g., "average publication year"), or sort by different criteria without manually reorganizing your data.

| Title | Author | Genre | Year | Read |
|-------|--------|-------|------|------|
| The Great Gatsby | F. Scott Fitzgerald | Fiction | 1925 | Yes |
| Sapiens | Yuval Noah Harari | Non-fiction | 2011 | Yes |
| Project Hail Mary | Andy Weir | Science Fiction | 2021 | No |
| Atomic Habits | James Clear | Self-help | 2018 | Yes |
| The Midnight Library | Matt Haig | Fiction | 2020 | No |

Another common example is tracking your **monthly expenses**. Each row could be a transaction with columns for the date, category (groceries, utilities, entertainment), amount, and description. With a DataFrame, you can quickly sum up spending by category, identify trends over time, or export the data to Excel for further analysis.

| Date | Category | Amount | Description |
|------|----------|--------|-------------|
| 2026-03-01 | Groceries | 45.50 | Weekly shopping |
| 2026-03-03 | Utilities | 120.00 | Electricity bill |
| 2026-03-05 | Entertainment | 15.99 | Movie ticket |
| 2026-03-08 | Groceries | 52.30 | Weekly shopping |
| 2026-03-15 | Utilities | 45.00 | Internet bill |

Or consider maintaining a **fitness log**: rows represent workout sessions with columns for the date, exercise type, duration, and calories burned. A DataFrame makes it trivial to calculate your average workout duration, count sessions per month, or visualize your progress over time.

In all these cases, you could use a spreadsheet, but a `DataFrame` (especially when combined with `Python`'s data analysis libraries) gives you powerful tools to organize, filter, analyze, and visualize your personal data programmatically.

| Date | Exercise | Duration (min) | Calories Burned |
|------|----------|----------------|-----------------|
| 2026-03-20 | Running | 30 | 350 |
| 2026-03-21 | Cycling | 45 | 420 |
| 2026-03-22 | Swimming | 40 | 380 |
| 2026-03-23 | Running | 25 | 280 |
| 2026-03-24 | Gym (weights) | 60 | 320 |

Let's create a table containing the artworks and their corresponding prices. A `DataFrame` can be instanciated from a dictionary whose keys are the column names and the values are the series. Because it is like a table, the number of entries in each column has to be equal.

In [ ]:
df = pd.DataFrame({
    "Artwork" : artworks,
    "Price" : prices
});
df

---

🗣 **Hint:** It is customary to use the suffix `df` to store a `DataFrame`. It is however better to choose a more evocative and explicit name such as `df_artwork_prices`.

---

The table we created has two dimensions and can be seen as a collection of series (i.e. the columns). The series (columns) can be accessed from their labels using the notation `df['label']`:

In [ ]:
df["Artwork"]

In [ ]:
df["Price"]

You can also select parts of the table which creates a *view* of it. We can select only one column but notice that **we do not get a column** BUT **a table with only one column**. Where 

```python
df["Price"]
```

is a column,


In [ ]:
df[["Price"]]

is a (view) of a table with one column. Look at the different data types:

In [ ]:
print(type(df))
print(type(df["Price"]))
print(type(df[["Price"]]))

You can then access each of the values in the column by specifying its column label and then its row index. Here is the
artwork in the second row (index 1):

In [ ]:
df['Artwork'][1]

And the price value in the fourth line (index 3):

In [ ]:
df["Price"][3]

This does not work with a table. But you can use the `iloc` (integer location) method to access the values of a row by specifying its index:

In [ ]:
df.iloc[1]

---

🗣 **Hint:** In computer science we always start counting indices from 0!

---

If you have a certain column with unique values, you can define them as index. Like this:

```python
df = pd.DataFrame({
    "Artwork" : artworks,
    "Price" : prices
}, index=unique_values);
```

Then you can use `loc` to get a row or multiple rows addressing them by values contained in `unique_values`.
Furthermore, the addressing syntax is very powerful. If you use `[a, b]` then `a` represents the rows and `b` the colums.
For example,

In [ ]:
df.iloc[:, 0]

selects all rows (`:`) but only the first column `0`. 

The `:` symbol is used to select all rows or all columns, but it can also be used to select a range of rows or columns.
For example, to select the rows from the second to the fourth, you can use the notation `df[1:4]`:

In [ ]:
df.iloc[1:4]

The `:` notation can also be used with `iloc` to select an arbitrary range of rows and columns:

In [ ]:
df.iloc[1:4, 1:]

You can even select specific rows and columns:

In [ ]:
df.iloc[[0,2], :]

`1:` means "select everything past the first (column)"

**Remark:**

```python
df["Artwork"]
```

selects the **column** named `"Artwork"`

```python
df.loc["Artwork"]
```

selects the **row** with the label `"Artwork"` (which does not work in our example, since there is no row with such a name).

### 9.3 Metadata and Statistics of a Table

We're now using `Pandas` to extract some metadata
and statistics from our data.

Firstly, the size of the table:

In [ ]:
df.shape # return (nb of rows, nb of columns)

The columns labels:

In [ ]:
df.columns

The number of rows:

In [ ]:
len(df)

Some general information on column types and memory usage:

In [ ]:
df.info()

The average of each column containing numerical values (in this case only prices):

In [ ]:
# we tell pandas to only look at numeric values
# axis = 0 is optional because it is the default and it means compute whatever you want to compute 
# along the rows i.e. that we get one value for each column!
df.mean(numeric_only = True, axis = 0) 

Standard deviations:

In [ ]:
df.std(numeric_only = True)

Medians:

In [ ]:
df.median(numeric_only = True)

The `describe` method provides a more exhaustive summary and statistics on each column:

In [ ]:
df.describe()

To add a new column to the table, you can use the following syntax:
```python
df['new_column'] = values
```

Let's add a column with the year of creation of each artwork: 

In [ ]:
df["Year"] = [2023,
              1903, 
              1904,
              1964, 
              1986,
              1961,
              1940,
              1970]
df

For fun we could compute the sum along `axis = 1` i.e. for each row (which makes no sense):

In [ ]:
df.sum(numeric_only = True, axis = 1)

In [ ]:
df.describe()

### 9.4 Basic operations on data

Pandas allows you to perform operations on data tables. 

Let's first transform the table so that one of the columns is used as indices, for example the artwork names:

In [ ]:
df_tmp = df.set_index("Artwork")

Each row can now be identified by a name that is equal to the name of the artwork. It is now possible to access a price value using the name directly as a line index:

In [ ]:
df_tmp['Price']["Patchwork Quilt"]

**Remark:**

```python
df_tmp['Price']["Patchwork Quilt"]
```

is equal to

```python
df_tmp['Price'].loc["Patchwork Quilt"]
```

Selection works also by masking, i.e. by generating a `Series` of `bool` that indicate to consider a row if the value is `True` or to filter it out if the value is `False`. Combining this with the power of *component-wise* operations makes `pandas` so powerful.

Let us now select all rows whose price is greater than 1 million of $:

In [ ]:
df[df["Price"] > 1000]

`df["Price"] > 1000` is a `Series` of `True` and `False` values.

In [ ]:
df["Price"] > 1000

Do you understand the above expression? Let's decompose it...

It is important to understand that operations are **vectorised** on all elements of a `Series`. So, if we write `prices + 100`, this adds 100 to all the values in the series:

In [ ]:
prices + 100

Similarly, if we write `prices == 5000`, it returns a series of Booleans, each one indicating whether the corresponding value is equal to $5000$ or not:

In [ ]:
prices == 5000

Let's now store the series of artworks whose price is greater than $1 million in a new variable:

In [ ]:
costMoreThan1000 = prices > 1000
costMoreThan1000

Indexing a `DataFrame` by a series of booleans will extracts the rows for which the series contains `True`:

In [ ]:
df[costMoreThan1000]

This is why 

```python
df[df["Price"] > 1000]
````

selects the rows for which the price is greater than 1000k of US dollars (1 million).

---

## 10 A Closer Look at the MoMA's Artist Curation

Now you've learned the basics to manipulate data with `pandas` (and a little bit of `numpy`), let's move on to a real-world problem!

The **Museum of Modern Art (MoMA)**, established in 1929, is a museum located in Midtown Manhattan, New York City. It houses an extensive collection that has grown to almost 200,000 artworks from around the globe, covering the last 150 years.

The file `moma_artist.csv` contains information about artists including: their name, nationality, gender, birth and death year.

The file `moma_artwork.csv`contains information about artworks including: their title, artist name, the date of acquisition, and the category of the artwork (painting, sculpture, etc.)

Our analysis will try to answer the following research questions:

+ **RQ1.** What are the demographic characteristics of the artists curated by the MoMA?
+ **RQ2.** Did the artist curation of the MoMA shifted since its creation?
+ **RQ3.** In particular, how has the proportion of international and women artists changed?

Data frames can be loaded from comma-separated values (CSV) files using the `read_csv` method:

In [ ]:
df_artists = pd.read_csv("data/moma_artist.csv") 
df_artists

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 10.1:** How many rows and columns does the dataset contain?

---

_Type your answer here, replacing this text._

In [ ]:
nrows = ...
ncols = ...

In [ ]:
grader.check("q101")

<!-- END QUESTION -->

The column `Artist ID` assigns a unique identifier to each individual, in case homonyms exist in the dataset.

In [ ]:
df_artists["Artist ID"].nunique()

### 10.1 Analyzing Artists' Nationality

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 10.2:** What is the most common artist's nationalities in this dataset? Use the `value_counts` method on the `Nationality` serie extracted from the dataframe.

---

_Type your answer here, replacing this text._

In [ ]:
...

<!-- END QUESTION -->

Visualization is of crucial importance in data analysis. The `seaborn` package is an extension of `matplotlib` that provides a high-level interface for drawing attractive and informative statistical graphics.

If you are not sure what would be a good visualization for your data, you can browse the excellent [Python graph gallery](https://python-graph-gallery.com/) that provides code snippets (using `matplotlib` or `seaborn`) to generate the plots.

Let's use an horizontal barplot to visualize the most common artist's nationalities:


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configure our plotting engine to get nice visualiziations
sns.set_theme(style="whitegrid")
sns.set_context("talk", font_scale=0.8)
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (10, 6)

For visibility let's only display the top 10 artists' nationalities:

In [ ]:
top10Nationalities = df_artists["Nationality"].value_counts().head(10)
top10Nationalities

In [ ]:
sns.barplot(x=top10Nationalities, y=top10Nationalities.index)

### 10.2 Analyzing Artists' Gender

Dataset often contain incorrect or missing values.
For example, let's look at artists' gender:

In [ ]:
gender = df_artists["Gender"].value_counts()
gender

It becomes obvious that there is an inconsistency in the format of the labels. Let's fix this by converting all labels to lowercase:

In [ ]:
df_artists["Gender"] = df_artists["Gender"].str.lower()
gender = df_artists["Gender"].value_counts()
gender

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 10.3:** Visualize the gender distribution of artists that were curated at the MoMA using the visualization of your choice.

---

In [ ]:
...

<!-- END QUESTION -->

### 10.3 Birth Date of Artists

Let's have a look at the birth date of the artists curated at the MoMA, using an histogram:

In [ ]:
num_bins = int(df_artists['Birth Year'].max() - df_artists['Birth Year'].min() + 1)
sns.histplot(data=df_artists, x='Birth Year', bins=num_bins)

Now let's visualize the birth year distribution of artists, grouped by gender. We assign the `Gender` column to the `hue` (color) parameter of the `sns.histplot` function, and decide to stack the bars by setting the `multiple` parameter to `stack`. Furthermore, we cumulate (`cumulative=True`) the values!

In other word, the histogram has the same shape as before but now we have colors for artists' gender.

In [ ]:
sns.histplot(data=df_artists, x='Birth Year',  hue="Gender", multiple="stack", bins=num_bins, cumulative=True)

It is non-trivial to interpret gender balance from this plot because:

- the number of artists per band of birth years birth years also varies;
- there is a lot of bins (band of birth years), i.e. the histogram is too granular;

Let's instead plot the proportion of female artists as their birth year increases. We can do this by using the `sns.histplot` function with the `stat` parameter set to `density` and the `multiple` parameter set to `fill`.

In [ ]:
sns.histplot(data=df_artists, x='Birth Year', hue="Gender", stat="density", multiple="fill", bins=25)

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 10.4:** Describe the visualization above.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 10.5:** Give possible interpretations of the visualization above. Keep in mind that interpretations are additional hypotheses that need to be verified with further analysis. They are however essential to guide future analysis and is part your role as a data analyst or researcher.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## 11 Analyzing the Artworks Acquired by the MoMA

The `moma_artist.csv` file contains limited information about the curation of the MoMA.
In particular, we do not have information at the artwork level or the date of acquisition of the pieces.

Let us now import the `moma_artwork.csv` file to get more information about the artworks.

In [ ]:
df_artworks = pd.read_csv("data/moma_artwork.csv")
df_artworks.head(7)

To further shed light on our question about gender balance in the curation of the MoMA, we can now analyze their curation at the level of artworks, and in particular the acquisition year.

<!-- BEGIN QUESTION -->

---
    
🖍 **Exercise 11.1:** What is the most recent acquisitions in the present dataset? Use the `sort_values` method to list the artworks by descending order of acquisition date.

---

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 11.2:** What is the most ancient acquisition present in the dataset? What can you say about it?

---

_Type your answer here, replacing this text._

In [ ]:
...

<!-- END QUESTION -->

Let's delete the row corresponding to the artwork with the acquisition date 1216 using the `drop` method and the `index` parameter:

In [ ]:
df_artworks = df_artworks.drop(index=128443)

We want to analyze artists' gender when their work are acquired by the MoMA. The problem is that `moma_artwork.csv` does not contain information about artists' gender.

We can however merge the two datasets, `moma_artist.csv` and `moma_artwork.csv`, on the basis of artist names and artists birth years.

In [ ]:
df_merged = pd.merge(df_artists, df_artworks, left_on=["Name", "Birth Year"], right_on=["Artist", "BirthYear"])
df_merged.head(5)

The `merge` method allows to merge two dataframes on the basis of common columns.
Now, each row corresponds to an artwork as in `moma_artwork.csv`, but the columns also have information about the artist as in `moma_artist.csv`.

### 11.1 Analyzing Artists' Nationality

Now that we have merged the two datasets, we can analyze the nationality of artists over time and at the artwork level.

Let's now implement an appropriate format and visualization to show the evolution of artists' nationality as a function of the acquisition year of artworks.

In [ ]:
# Clean data
df_clean = df_merged.dropna(subset=['Nationality', 'YearAcquired'])

# Identify the top 5 nationalities
top_nationalities = df_clean['Nationality'].value_counts().nlargest(5).index

# Group and recategorize nationalities
df_clean['GroupedNationality'] = df_clean['Nationality'].apply(lambda x: x if x in top_nationalities else 'Other')

# Group by YearAcquired and GroupedNationality and count the artworks
nationality_yearly = df_clean.groupby(['YearAcquired', 'GroupedNationality']).size()

# Convert the Series to a DataFrame for easier manipulation
nationality_yearly_df = nationality_yearly.unstack(fill_value=0)

# Cumulative total artworks acquired by nationality over years
cumulative_nationality = nationality_yearly_df.cumsum()

# Sort columns based on values in 2017
cumulative_nationality_sorted = cumulative_nationality.loc[:, cumulative_nationality.loc[2017].sort_values(ascending=False).index]
cumulative_nationality_sorted

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Plotting
plt.figure(figsize=(12, 6))
cumulative_nationality_sorted.plot(kind='area', stacked=True, ax=plt.gca())
plt.title("Cumulative artists' nationality from MoMA artworks acquisitions")
plt.xlabel('Year Acquired')
plt.ylabel('Total Artworks Acquired')
plt.legend(title='Nationality', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()  # Adjusts plot to ensure everything fits without overlap
plt.show()


<!-- BEGIN QUESTION -->

---

🖍 **Exercise 11.3:** Describe and interpret the visualization above.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 11.2 Analyzing Artists' Gender

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 11.4:** Repeat a similar analysis but for artists' gender as a function of artwork acquisition year. Use a stacked area plot as a visualization.

---

In [ ]:
...

In [ ]:
...

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

🖍 **Exercise 11.5:** Describe and interpret the visualization above.

---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

**Exercise 11.6:** Conclude about your analysis of the MoMA dataset and according to the research questions raised at the beginning. What are the limitations of your analysis and possible future work?


---

_Type your answer here, replacing this text._

<!-- END QUESTION -->

## Well done!

You've learned the basics of `pandas` and `seaborn`, two essential packages for data manipulation and visualization in Python.

In particular, you should now be able to:

 - create and manipulate `Series` and `DataFrame` objects;
 - extract metadata and statistics from a dataset;
 - perform basic operations on data, such as filtering and adding columns;
 - visualize data using `seaborn` and `matplotlib`;
 - merge two datasets on the basis of common columns;
 - comment and interpret the results of your analysis.

It is time to learn about machine learning in Python! Go to the next notebook:

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A1_introduction/4_linear_regression.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>